In [3]:
#######
# Function parse_trace_file
#   Wrangle and clean measurement strain data file 
#   Including some mangled data error catching
# Parameters:
#   path - file path to target load cell data (str)
# Returns:
#   Formatted dataframe (pandas Dataframe of form Measurement | Datetime)
#######
import pandas as pd
import os



def parse_trace_file(path):
    # Try to parse file
    try:
        # Read original CSV file
        dat = pd.read_csv(path, header=None, names=["Measure", "Datetime"], 
                          encoding="utf-8", encoding_errors="replace", on_bad_lines="skip", 
                          engine="python")
        
        # Convert measurement column to numeric strain values
        dat["Measure"] = pd.to_numeric(dat["Measure"], errors="coerce")            

        # Convert Unix timestamp to datetime
        dat["Datetime"] = pd.to_datetime(dat["Datetime"], unit='s', utc=False, errors="coerce")
        dat["Datetime"] = dat["Datetime"].dt.strftime('%Y-%m-%d %H:%M:%S')

        # We've possibly forced parsing of some malformed data
        #   ("replace" utf-8 encoding errors in read_csv(), "coerce" datetime errors in to_numeric() and to_datetime(), 
        # Now we need to clean that up.
        # Simply drop all points from the file where Measure has been coerced to NaN
        #   and where Datetime has been coerced to NaT
        dat = dat[~dat.Measure.isnull() & ~dat.Datetime.isnull()]
        
        return dat
    
    # Common errors
    except FileNotFoundError:
        warnings.warn("\nInput file not found. FileNotFoundError")
    except pd.errors.EmptyDataError:
        warnings.warn("\nError parsing empty file. EmptyDataError")
    except pd.errors.ParserError:
        warnings.warn("\nError parsing input file. ParserError")
    except Exception as e:
        warnings.warn("\nError parsing input file. {}".format(e))


In [4]:
########
# Load MOM DATA
#######
import pandas as pd
import os

print(pd.__version__)

file_path_MOM = "/Users/bobmauck/devel/Testing_Data/DL_06_11_2025_952.TXT"   # get_user_file()# Get user file path
file_path_MOM = "/Users/bobmauck/devel/Testing_Data/DL_07_09_2025_933.TXT"
file_path_MOM = "/Users/bobmauck/devel/Testing_Data/DL_02_19.TXT"
# file_path_MOM = "/Users/bobmauck/devel/Testing_Data/HZ51_DL_02_18.TXT"
file_path_MOM = "/Volumes/MOM_TEST/DL260315.TXT"
filename = os.path.basename(file_path_MOM)
df_MOM_All = parse_trace_file(file_path_MOM)
df_MOM_All.head(3)

2.3.3


,Measure,Datetime
0,8378079,2026-03-15 18:20:52
1,8377993,2026-03-15 18:20:52
2,8377963,2026-03-15 18:20:52


In [10]:
import pandas as pd
import numpy as np

def estimate_hz_from_sparse_datetime(df, CountN=10000, datetime_col="Datetime"):
    """
    Estimate sample rate (Hz) from a dataframe where only some rows have timestamps.

    Method:
    - Use up to CountN rows.
    - Keep rows with valid datetime values in `datetime_col`.
    - For each consecutive timestamp pair:
        hz_i = (# rows between timestamps) / (seconds between timestamps)
    - Return weighted average Hz over all valid intervals.

    Returns:
        float (estimated Hz)
    """
    if datetime_col not in df.columns:
        raise ValueError(f"Missing column: {datetime_col}")

    d = df.iloc[:CountN].copy()

    # Parse datetimes; invalid/missing become NaT
    d["_dt"] = pd.to_datetime(d[datetime_col], errors="coerce")

    ts = d["_dt"].dropna()
    if len(ts) < 2:
        raise ValueError("Need at least 2 valid timestamps within CountN rows.")

    idx = ts.index.to_numpy()
    t = ts.to_numpy()

    # Rows and seconds between consecutive timestamped rows
    rows_between = np.diff(idx).astype(float)
    secs_between = np.diff(t).astype("timedelta64[ns]").astype(np.float64) / 1e9

    valid = secs_between > 0
    if not np.any(valid):
        raise ValueError("No positive time intervals found between timestamps.")

    hz = rows_between[valid].sum() / secs_between[valid].sum()
    return float(hz)
estimated_hz = estimate_hz_from_sparse_datetime(df_MOM_All, CountN=10000, datetime_col="Datetime")
print(f"Estimated sample rate (Hz): {estimated_hz:.2f}")

Estimated sample rate (Hz): 0.19


In [11]:
hz = estimate_hz_from_sparse_datetime(df_MOM_All, CountN=10000)
print(f"Estimated sample rate: {hz:.3f} Hz")


Estimated sample rate: 0.188 Hz


In [7]:
import pandas as pd

def estimate_hz_v02(df, datetime_col="Datetime", CountN=None):
    """
    Simple Hz estimate:
    - starttime = first valid datetime
    - endtime   = last valid datetime
    - lines     = total rows used
    - hz        = lines / elapsed_seconds
    """
    if datetime_col not in df.columns:
        raise ValueError(f"Missing column: {datetime_col}")

    d = df if CountN is None else df.iloc[:CountN]
    lines = len(d)
    if lines == 0:
        raise ValueError("DataFrame is empty.")

    dt = pd.to_datetime(d[datetime_col], errors="coerce").dropna()
    if dt.empty:
        raise ValueError("No valid timestamps found.")

    starttime = dt.iloc[0]
    endtime = dt.iloc[-1]
    elapsed_seconds = (endtime - starttime).total_seconds()

    if elapsed_seconds <= 0:
        raise ValueError("Elapsed time must be > 0 seconds.")

    hz = lines / elapsed_seconds

    print(f"starttime: {starttime}")
    print(f"endtime:   {endtime}")
    print(f"lines:     {lines}")
    print(f"hz:        {hz:.1f}")

    return hz


In [6]:
import pandas as pd

def estimate_hz_v031(df, datetime_col="Datetime", CountN=None):
    """
    v03:
    1) Build a new df with one row per Datetime and the count of lines for that Datetime.
    2) Exclude first and last rows of that grouped df.
    3) For each remaining row, compute:
         hz = lines_in_that_datetime / seconds_to_next_datetime
    4) Return mean hz.
    """
    if datetime_col not in df.columns:
        raise ValueError(f"Missing column: {datetime_col}")

    d = df if CountN is None else df.iloc[:CountN]
    if d.empty:
        raise ValueError("Input DataFrame is empty.")

    # Parse datetime and keep valid rows
    dt = pd.to_datetime(d[datetime_col], errors="coerce")
    valid = pd.DataFrame({"Datetime": dt}).dropna()

    if valid.empty:
        raise ValueError("No valid Datetime values found.")

    # One row per Datetime, with line counts
    per_dt = (
        valid.groupby("Datetime", sort=True)
        .size()
        .reset_index(name="Lines")
    )

    # Need enough rows after removing first/last
    if len(per_dt) < 4:
        raise ValueError("Need at least 4 unique Datetime values for v03.")

    core = per_dt.iloc[1:-1].copy()  # remove first and last rows
    core["NextDatetime"] = core["Datetime"].shift(-1)
    core["ElapsedSec"] = (core["NextDatetime"] - core["Datetime"]).dt.total_seconds()

    # Last core row has no next row; drop invalid/zero/negative intervals
    core = core.dropna(subset=["ElapsedSec"])
    core = core[core["ElapsedSec"] > 0].copy()

    if core.empty:
        raise ValueError("No positive elapsed intervals available to compute Hz.")

    
    core["Hz"] = core["Lines"] / core["ElapsedSec"]
    mean_hz = core["Hz"].mean()
    STD_hz = core["Hz"].std()

    print(f"Mean Hz (v03): {mean_hz:.6f}")
    print(f"STD Hz (v03):  {STD_hz:.6f}")
    return mean_hz, core




In [ ]:
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt

def estimate_hz_v03(df, datetime_col="Datetime", CountN=None, plot=False):
    if datetime_col not in df.columns:
        raise ValueError(f"Missing column: {datetime_col}")

    d = df if CountN is None else df.iloc[:CountN]
    if d.empty:
        raise ValueError("Input DataFrame is empty.")

    dt = pd.to_datetime(d[datetime_col], errors="coerce")
    valid = pd.DataFrame({"Datetime": dt}).dropna()
    if valid.empty:
        raise ValueError("No valid Datetime values found.")

    per_dt = valid.groupby("Datetime", sort=True).size().reset_index(name="Lines")
    if len(per_dt) < 4:
        raise ValueError("Need at least 4 unique Datetime values for v03.")

    core = per_dt.iloc[1:-1].copy()  # drop first and last
    core["NextDatetime"] = core["Datetime"].shift(-1)
    core["ElapsedSec"] = (core["NextDatetime"] - core["Datetime"]).dt.total_seconds()
    core = core[(core["ElapsedSec"].notna()) & (core["ElapsedSec"] > 0)].copy()
    if core.empty:
        raise ValueError("No positive elapsed intervals available to compute Hz.")

    core["Hz"] = core["Lines"] / core["ElapsedSec"]

    # Ensure numeric column for plotting
    per_dt["Hz"] = np.nan
    per_dt.loc[core.index, "Hz"] = core["Hz"].to_numpy()

    mean_hz = core["Hz"].mean()
    print(f"Mean Hz (v03): {mean_hz:.6f}")

    if plot:
        plot_df = per_dt.dropna(subset=["Hz"])
        plt.figure(figsize=(7, 4))
        plt.scatter(plot_df.index.to_numpy(), plot_df["Hz"].to_numpy(), s=12)
        plt.title("Per-interval Hz")
        plt.xlabel("per_dt row index")
        plt.ylabel("Hz")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    return mean_hz, per_dt



SyntaxError: invalid syntax (1865354925.py, line 3)

In [9]:
hz = estimate_hz_v02(df_MOM_All)          # whole df
# hz = estimate_hz_v02(df_MOM_All, CountN=10000)  # optional cap

print("Averaging set at 60, previously was 80")
print('Version 3...')
mean_hz, per_dt = estimate_hz_v03(df_MOM_All, CountN=10000, plot=True)

print("DT value: ")
print (per_dt.mean())


starttime: 2026-03-15 18:20:52
endtime:   2026-03-16 06:59:55
lines:     2560200
hz:        56.2
Averaging set at 60, previously was 80
Version 3...


NameError: name 'estimate_hz_v03' is not defined